In [ ]:
# Ячейка 1: Постановка
"""
## Выездные бригады по районам

4 бригады нужно распределить по 4 районам.
Время в пути (минут):
        Р1  Р2  Р3  Р4
Бригада A: 20  30  25  40
Бригада B: 35  15  30  25
Бригада C: 40  20  35  30
Бригада D: 25  35  20  15

Задание:
1. Решить через linear_sum_assignment
2. Проверить ограничения
3. Проверить через linprog
4. Объяснить назначения
"""

In [ ]:
# Ячейка 2: Решение через венгерский алгоритм
import numpy as np
from scipy.optimize import linear_sum_assignment

cost = np.array([
    [20, 30, 25, 40],
    [35, 15, 30, 25],
    [40, 20, 35, 30],
    [25, 35, 20, 15]
])

row_ind, col_ind = linear_sum_assignment(cost)

print("Назначения (бригада -> район):")
total = 0
for i, j in zip(row_ind, col_ind):
    print(f"Бригада {chr(65+i)} -> Район {j+1}: {cost[i,j]} мин")
    total += cost[i, j]
print(f"Общее время: {total} мин")

# Проверка ограничений
X = np.zeros_like(cost)
X[row_ind, col_ind] = 1
assert X.sum(axis=1).all() == 1, "Не каждый исполнитель назначен"
assert X.sum(axis=0).all() == 1, "Не каждая задача назначена"
print("Проверки пройдены")

In [ ]:
# Ячейка 3: LP-проверка через linprog
from scipy.optimize import linprog

n = 4
c = cost.flatten()

A_eq = []
for i in range(n):
    row = np.zeros(n*n)
    row[i*n:(i+1)*n] = 1
    A_eq.append(row)
for j in range(n):
    row = np.zeros(n*n)
    row[j::n] = 1
    A_eq.append(row)

b_eq = [1]*n + [1]*n
result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0, 1), method='highs')

X_lp = result.x.reshape(n, n)
print("LP решение совпадает:", np.allclose(X, X_lp.round()))
print(f"Стоимость LP: {result.fun:.0f} мин")

In [ ]:
# Ячейка 4: Вывод
"""
Бригада A -> Район 1 (20 мин) - ближайший
Бригада B -> Район 2 (15 мин) - ближайший
Бригада C -> Район 3 (35 мин)
Бригада D -> Район 4 (15 мин) - ближайший
Общее время: 85 минут.
Венгерский алгоритм и LP дают одинаковый результат.
"""